# 07. 변이 유형 피처

## 배경
- 06에서 행 정규화가 CV를 낮춤 → "변이 개수" 자체가 유용한 신호로 보임
- train 변이의 26%가 침묵 변이 (단백질이 바뀌지 않아 잡음일 가능성)

## 가설
- H2: 행 단위 변이 개수 피처(유형별)를 추가하면 CV가 오른다
- H3: 침묵 변이만 있는 칸을 0으로 처리(non-silent)하면 잡음이 줄어 CV가 오른다
- H4: 절단 변이(stop, frameshift) 여부를 유전자별로 따로 주면 CV가 오른다

## 검증 방법
- 모델 고정: LR(C=0.05, class_weight='balanced'), 공통 CV
- A(기준) → B(+행 피처) → C(non-silent + 행 피처) → D(C + 절단 행렬) 순서로 하나씩 추가
- 판단 기준: 0.3838 대비 +0.01 이상이면 채택
- 모든 피처는 행 단위 계산 → Data Leakage 아님. 스케일러는 fold 안에서만 fit

In [1]:
import sys; sys.path.append('..')
import re, time, numpy as np, pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from src.config import SEED, WT
from src.data import load_data, get_gene_cols
from src.features import to_binary
from src.validation import seed_everything, encode_target, run_cv, save_submission

seed_everything()
train, test, submission = load_data()
genes = get_gene_cols(train)
y, le = encode_target(train)

#### 변이 유형 파싱 함수

In [2]:
SILENT = re.compile(r'([A-Z])\d+\1')      # R895R
MISSENSE = re.compile(r'[A-Z]\d+[A-Z]')   # G12D
TYPES = ['silent', 'missense', 'stop', 'fs', 'other']

def mut_type(t):
    if SILENT.fullmatch(t):   return 0
    if 'fs' in t:             return 3
    if '*' in t:              return 2
    if MISSENSE.fullmatch(t): return 1
    return 4

def mutation_features(df, genes):
    """한 행 안에서만 계산 → test에 적용해도 누수 아님.
    Returns: X_any, X_nonsilent, X_trunc (유전자 행렬), stats (행 피처)"""
    V = df[genes].to_numpy()
    n, m = V.shape
    nonsilent = np.zeros((n, m), np.int8)
    trunc = np.zeros((n, m), np.int8)
    cnt = np.zeros((n, len(TYPES)), np.float32)

    rows, cols = np.nonzero(V != WT)
    for i, j in zip(rows, cols):
        for t in V[i, j].split():          # 한 칸에 여러 변이 → 공백 구분
            k = mut_type(t)
            cnt[i, k] += 1
            if k != 0:      nonsilent[i, j] = 1
            if k in (2, 3): trunc[i, j] = 1

    X_any = to_binary(df, genes).astype(np.float32)
    X_ns = pd.DataFrame(nonsilent, index=df.index, columns=genes, dtype=np.float32)
    X_tr = pd.DataFrame(trunc, index=df.index, columns=[f'{g}__trunc' for g in genes], dtype=np.float32)

    stats = pd.DataFrame(np.log1p(cnt), index=df.index, columns=[f'n_{t}_log' for t in TYPES])
    stats['n_gene_log'] = np.log1p(X_any.sum(axis=1))
    stats['silent_ratio'] = cnt[:, 0] / np.maximum(cnt.sum(axis=1), 1)
    return X_any, X_ns, X_tr, stats

t = time.time()
tr_any, tr_ns, tr_tr, tr_stats = mutation_features(train, genes)
te_any, te_ns, te_tr, te_stats = mutation_features(test, genes)
print(f'{time.time()-t:.0f}s', tr_stats.describe().T[['mean', '50%', 'max']].round(2))

4s                 mean   50%   max
n_silent_log    1.53  1.39  8.19
n_missense_log  2.40  2.30  8.70
n_stop_log      0.62  0.69  5.82
n_fs_log        0.57  0.69  4.94
n_other_log     0.02  0.00  2.40
n_gene_log      2.76  2.71  7.78
silent_ratio    0.22  0.23  1.00


#### 셀 A~D 비교

In [3]:
STAT_COLS = list(tr_stats.columns)

def make_lr(C=0.05, scale_stats=True):
    # fold마다 '새' 모델을 만들어야 result['models']에 fold별 모델이 따로 남음
    def new_lr():
        return LogisticRegression(C=C, max_iter=3000, class_weight='balanced', random_state=SEED)
    if not scale_stats:
        return new_lr
    # 행 피처만 표준화 (fold 학습 부분에서만 fit), 이진 행렬은 그대로
    return lambda: make_pipeline(
        ColumnTransformer([('sc', StandardScaler(), STAT_COLS)], remainder='passthrough'),
        new_lr())

sets = {
    'A_any':               (tr_any, None),
    'B_any+stats':         (pd.concat([tr_any, tr_stats], axis=1), True),
    'C_nonsilent+stats':   (pd.concat([tr_ns, tr_stats], axis=1), True),
    'D_ns+trunc+stats':    (pd.concat([tr_ns, tr_tr, tr_stats], axis=1), True),
}

rows = []
for name, (Xs, has_stats) in sets.items():
    t = time.time()
    r = run_cv(make_lr(scale_stats=bool(has_stats)), Xs, y, verbose=False)
    rows.append(dict(set=name, n_feat=Xs.shape[1], cv=r['cv_mean'], std=r['cv_std'],
                     oof=r['oof_score'], sec=round(time.time()-t)))
    print(rows[-1])
res7 = pd.DataFrame(rows)
res7

{'set': 'A_any', 'n_feat': 4384, 'cv': 0.3838190233315374, 'std': 0.007927521435212194, 'oof': 0.3850511311538694, 'sec': 3}
{'set': 'B_any+stats', 'n_feat': 4391, 'cv': 0.3941550256100893, 'std': 0.012967645342199604, 'oof': 0.3953846387982448, 'sec': 5}
{'set': 'C_nonsilent+stats', 'n_feat': 4391, 'cv': 0.386883442100076, 'std': 0.013980997377422269, 'oof': 0.3878421697375923, 'sec': 3}
{'set': 'D_ns+trunc+stats', 'n_feat': 8775, 'cv': 0.3982365708151899, 'std': 0.011663035958006978, 'oof': 0.39944322552988776, 'sec': 7}


,set,n_feat,cv,std,oof,sec
0,A_any,4384,0.383819,0.007928,0.385051,3
1,B_any+stats,4391,0.394155,0.012968,0.395385,5
2,C_nonsilent+stats,4391,0.386883,0.013981,0.387842,3
3,D_ns+trunc+stats,8775,0.398237,0.011663,0.399443,7


## 결과 1: A~D 비교 (LR C=0.05, 공통 CV)

| 조합 | 피처 수 | CV (± std) | 비교 | 가설 판단 |
| :--- | --: | :-: | :-: | :--- |
| A_any | 4,384 | 0.3838 ± 0.0079 | — | 06 최고값과 동일 → 코드 정상 |
| B_any+stats | 4,391 | 0.3942 ± 0.0130 | A 대비 **+0.010** | **H2 채택** |
| C_nonsilent+stats | 4,391 | 0.3869 ± 0.0140 | B 대비 **−0.007** | **H3 기각** |
| D_ns+trunc+stats | 8,775 | **0.3982** ± 0.0117 | C 대비 **+0.011** | **H4 채택** |

### 해석
- **H2 (행 피처)**: 유형별 변이 개수를 추가하면 CV가 오름
  → 06의 "변이 개수가 유용한 신호" 해석과 일치
- **H3 (침묵 변이 제거)**: 침묵 변이만 있는 칸을 0으로 바꾸면 손해
  → 침묵 변이도 신호. 단백질은 바뀌지 않지만, 암종마다 변이가 생기는 양상이 달라
    침묵 변이에도 그 흔적이 남는 것으로 보임
- **H4 (절단 변이 행렬)**: 유전자별 stop/frameshift 여부를 따로 주면 CV가 오름
  → 같은 유전자라도 기능을 잃는 변이인지가 암종 구분에 도움이 됨

### 주의
- 표준편차가 0.008 → 0.012~0.014로 커짐. A 대비 D의 개선폭(+0.014)은 표준편차 1개 수준
  → 확실한 개선이라기보다 LB로 확인할 가치가 있는 수준

### 남은 확인
- D는 H3에서 손해였던 non-silent 행렬 위에 절단 행렬을 얹은 조합
- **E (침묵 변이 포함 행렬 + 절단 행렬 + 행 피처)**를 아직 검증하지 않음
  → H3 결과로 보면 D보다 좋을 가능성이 큼. 피처가 늘었으므로 C도 함께 재확인
  

#### E 조합 + C 재확인

In [4]:
sets2 = {
    'D_ns+trunc+stats':  pd.concat([tr_ns,  tr_tr, tr_stats], axis=1),
    'E_any+trunc+stats': pd.concat([tr_any, tr_tr, tr_stats], axis=1),
}
rows = []
for name, Xs in sets2.items():
    for C in [0.03, 0.05, 0.1]:
        r = run_cv(make_lr(C=C), Xs, y, verbose=False)
        rows.append(dict(set=name, C=C, cv=r['cv_mean'], std=r['cv_std'], oof=r['oof_score']))
        print(rows[-1])
res7b = pd.DataFrame(rows).sort_values('cv', ascending=False)
res7b

{'set': 'D_ns+trunc+stats', 'C': 0.03, 'cv': 0.3842114470368743, 'std': 0.008473059901042284, 'oof': 0.3854924587434553}
{'set': 'D_ns+trunc+stats', 'C': 0.05, 'cv': 0.3982365708151899, 'std': 0.011663035958006978, 'oof': 0.39944322552988776}
{'set': 'D_ns+trunc+stats', 'C': 0.1, 'cv': 0.40739786758478225, 'std': 0.012189743248884163, 'oof': 0.4086663608903879}
{'set': 'E_any+trunc+stats', 'C': 0.03, 'cv': 0.3997679268873143, 'std': 0.012762506899182163, 'oof': 0.4007720254282978}
{'set': 'E_any+trunc+stats', 'C': 0.05, 'cv': 0.4099226436585365, 'std': 0.015259309098400007, 'oof': 0.4107394148597328}
{'set': 'E_any+trunc+stats', 'C': 0.1, 'cv': 0.4188079820328614, 'std': 0.0173387766032759, 'oof': 0.41992786679977656}


,set,C,cv,std,oof
5,E_any+trunc+stats,0.10,0.418808,0.017339,0.419928
4,E_any+trunc+stats,0.05,0.409923,0.015259,0.410739
2,D_ns+trunc+stats,0.10,0.407398,0.012190,0.408666
3,E_any+trunc+stats,0.03,0.399768,0.012763,0.400772
1,D_ns+trunc+stats,0.05,0.398237,0.011663,0.399443
0,D_ns+trunc+stats,0.03,0.384211,0.008473,0.385492


In [5]:
rows = []
Xe = sets2['E_any+trunc+stats']
for C in [0.15, 0.2, 0.3, 0.5]:
    r = run_cv(make_lr(C=C), Xe, y, verbose=False)
    rows.append(dict(set='E_any+trunc+stats', C=C, cv=r['cv_mean'], std=r['cv_std'], oof=r['oof_score']))
    print(rows[-1])
res7b = pd.concat([res7b, pd.DataFrame(rows)]).sort_values('cv', ascending=False)
res7b.head()

{'set': 'E_any+trunc+stats', 'C': 0.15, 'cv': 0.41671283774756096, 'std': 0.01621111518546806, 'oof': 0.417507172534924}
{'set': 'E_any+trunc+stats', 'C': 0.2, 'cv': 0.4156627977890765, 'std': 0.01230100712358336, 'oof': 0.4164418384671756}
{'set': 'E_any+trunc+stats', 'C': 0.3, 'cv': 0.40658090336905195, 'std': 0.012254038374531524, 'oof': 0.40711810844158386}
{'set': 'E_any+trunc+stats', 'C': 0.5, 'cv': 0.3975973006857325, 'std': 0.008622867203191178, 'oof': 0.39871552952361855}


,set,C,cv,std,oof
5,E_any+trunc+stats,0.10,0.418808,0.017339,0.419928
0,E_any+trunc+stats,0.15,0.416713,0.016211,0.417507
1,E_any+trunc+stats,0.20,0.415663,0.012301,0.416442
4,E_any+trunc+stats,0.05,0.409923,0.015259,0.410739
2,D_ns+trunc+stats,0.10,0.407398,0.012190,0.408666


#### 제출 파일 생성 코드

In [6]:
best = res7b.iloc[0]
print(best)   # set=E_any+trunc+stats, C=0.1 인지 확인
r = run_cv(make_lr(C=0.1), sets2['E_any+trunc+stats'], y,
           pd.concat([te_any, te_tr, te_stats], axis=1))
save_submission(submission, r['test_proba'], le,
                name="eunji_lr_any_trunc_stats_C0.1", cv_score=r['cv_mean'])

set    E_any+trunc+stats
C                    0.1
cv              0.418808
std             0.017339
oof             0.419928
Name: 5, dtype: object
[fold 0] Macro F1 = 0.4187
[fold 1] Macro F1 = 0.4243
[fold 2] Macro F1 = 0.4045
[fold 3] Macro F1 = 0.3985
[fold 4] Macro F1 = 0.4481
CV Macro F1 = 0.4188 ± 0.0173 (OOF 0.4199)
저장 완료: /Users/mac/web/dacon-cancer-subclass/submissions/1109_eunji_lr_any_trunc_stats_C0.1_0.419.csv


PosixPath('/Users/mac/web/dacon-cancer-subclass/submissions/1109_eunji_lr_any_trunc_stats_C0.1_0.419.csv')

## 결과

| 조합 | 피처 수 | C | CV (± std) | 0.3838 대비 |
| :--- | --: | :-: | :-: | :-: |
| A_any | 4,384 | 0.05 | 0.3838 ± 0.0079 | — |
| B_any+stats | 4,391 | 0.05 | 0.3942 ± 0.0130 | +0.010 |
| C_nonsilent+stats | 4,391 | 0.05 | 0.3869 ± 0.0140 | +0.003 |
| D_ns+trunc+stats | 8,775 | 0.1* | 0.4074 ± 0.0122 | +0.024 |
| **E_any+trunc+stats** | 8,775 | **0.1*** | **0.4188 ± 0.0173** | **+0.035** |

\* C 탐색 후 최적값. B, C는 C=0.05만 측정
- C 탐색 (E): 0.03 → 0.3998, 0.05 → 0.4099, **0.1 → 0.4188**, 0.15 → 0.4167, 0.2 → 0.4157, 0.3 → 0.4066, 0.5 → 0.3976
- 제출: E + C=0.1 → **LB 0.2706** (이전 최고 0.2739 대비 −0.003), CV−LB 차이 0.108 → **0.148로 확대**

## 결론
- **CV 기준** 가설 판단
  - **H2 채택**: 유형별 변이 개수(행 피처) 추가 시 +0.010
  - **H3 기각**: 침묵 변이를 빼면 하락 (B → C −0.007, E → D −0.011)
    → 침묵 변이도 암종별 변이 패턴을 담은 신호로 보임
  - **H4 채택**: 절단 변이(stop, frameshift) 유전자 행렬 추가 시 +0.011~0.016 (C=0.05 동일 조건)
  - 피처가 늘면서 최적 C가 0.05 → 0.1로 커짐, std도 0.008 → 0.017로 커짐
- **LB 기준: CV 개선(+0.035)이 LB로 옮겨가지 않음 (−0.003)**
  - 추정 원인: 변이 개수 피처가 train/test 분포 차이(test 변이 비율 약 2배)에 민감
  - 교훈: 이 대회에서는 CV만으로 피처를 고르면 위험. 변이 "개수"보다 "비율·유무" 기반 피처를 우선 검토

## 다음
- stats 제외(F: any+trunc), 비율 피처만 사용(G: any+trunc+ratio) 버전 CV 비교 → 진단용으로 1회 제출
- 05와 07 제출 파일의 예측 분포 비교 (변이 많은 클래스로 예측이 쏠렸는지)
- mutation_features()를 src/features.py로 옮겨 팀 공통 함수화
- 같은 피처로 LightGBM 재비교 → LR + LGBM 앙상블

-----

## 확인 1. 예측 분포 비교

### 배경
- E + C=0.1: CV 0.4188 (05 대비 +0.037) → LB 0.2706 (05 대비 −0.003)
- CV−LB 차이가 0.108 → 0.148로 확대
- test의 변이 칸 비율이 train의 약 2배로 보임 (진단 용도로만 참고, 튜닝 근거로 쓰지 않음)

### 가설
- H5: 변이 개수 피처(stats) 때문에, test에서 07(E) 모델의 예측이
  **변이가 많은 클래스 쪽으로 쏠렸다**
- H0: 05 → 07 사이 예측 분포의 변화는 클래스별 변이 수와 관계없다

### 검증 방법
- 05와 07 제출 파일의 예측 클래스 비율을 비교 (diff = 07 − 05)
- train에서 구한 클래스별 변이 유전자 수 중앙값(train_med_mut)과 diff의 관계를 확인
- 판단 기준
  - 예측이 늘어난 클래스(diff > 0)가 변이 많은 클래스, 줄어든 클래스가 변이 적은 클래스 → H5 채택
  - 관계가 뚜렷하지 않음 → H5 보류, 확인 2(F/G) 제출로 판단

### 누수 여부
- 모델 출력(제출 파일)과 train 통계만 비교 → test 데이터를 학습이나 튜닝에 쓰지 않음
- 이 결과로 정하는 것은 "어떤 종류의 피처를 피할지"라는 방향뿐, test에 맞춘 값 조정은 하지 않음

In [ ]:
# 예측 분포 비교
import pandas as pd
from src.config import SUBMISSION_DIR as S
subs = {'05_C0.1': '1109_eunji_lr_l0_C0.1_0.382.csv',
        '07_E':    '1109_eunji_lr_any_trunc_stats_C0.1_0.419.csv'}
dist = pd.DataFrame({k: pd.read_csv(S / v).SUBCLASS.value_counts(normalize=True) for k, v in subs.items()})
dist['train'] = train.SUBCLASS.value_counts(normalize=True)
dist['train_med_mut'] = train.assign(n=tr_any.sum(axis=1)).groupby('SUBCLASS').n.median()   # 클래스별 변이 수 중앙값 (train)
dist['diff'] = dist['07_E'] - dist['05_C0.1']
print(dist.sort_values('diff').round(3))


print('Spearman(diff, train_med_mut):', round(dist['diff'].corr(dist['train_med_mut'], method='spearman'), 3))

          05_C0.1   07_E  train  train_med_mut   diff
SUBCLASS                                             
COAD        0.225  0.074  0.036           23.0 -0.152
THYM        0.066  0.034  0.016            2.0 -0.032
PCPG        0.045  0.034  0.024            4.0 -0.011
OV          0.065  0.054  0.041           12.0 -0.011
PAAD        0.020  0.009  0.019           14.0 -0.011
BRCA        0.062  0.051  0.127            9.0 -0.010
SARC        0.018  0.011  0.032            9.0 -0.007
BLCA        0.014  0.008  0.017           54.0 -0.005
GBMLGG      0.022  0.017  0.074           11.0 -0.005
LAML        0.015  0.011  0.025            3.0 -0.005
CESC        0.029  0.024  0.025           20.0 -0.005
HNSC        0.034  0.029  0.036           31.0 -0.005
THCA        0.028  0.024  0.052            4.0 -0.004
STES        0.079  0.076  0.061           41.0 -0.003
KIRC        0.028  0.026  0.054           13.0 -0.002
LGG         0.035  0.034  0.037            8.0 -0.002
UCEC        0.028  0.027  0.

In [14]:
# train 대비 쏠림 배율
# 예측 비율 / train 비율: 1이면 train과 같은 비율, 3 이상이면 크게 쏠림
dist['ratio_05'] = (dist['05_C0.1'] / dist['train']).round(2)
dist['ratio_07'] = (dist['07_E']    / dist['train']).round(2)
cols = ['train', '05_C0.1', '07_E', 'ratio_05', 'ratio_07', 'train_med_mut']
print(dist.sort_values('ratio_07', ascending=False)[cols].head(8))
print('3배 이상 쏠린 클래스 | 05:', list(dist.index[dist.ratio_05 >= 3]), '| 07:', list(dist.index[dist.ratio_07 >= 3]))

             train   05_C0.1      07_E  ratio_05  ratio_07  train_med_mut
SUBCLASS                                                                 
LUAD      0.029673  0.013747  0.109191      0.46      3.68           40.5
SKCM      0.044509  0.025923  0.114690      0.58      2.58           85.5
TGCT      0.019997  0.035742  0.044776      1.79      2.24           16.0
THYM      0.015804  0.066379  0.034171      4.20      2.16            2.0
ACC       0.011611  0.009034  0.024745      0.78      2.13           24.0
COAD      0.035962  0.225452  0.073841      6.27      2.05           23.0
PCPG      0.023706  0.045169  0.034171      1.91      1.44            4.0
LUSC      0.028705  0.016496  0.039277      0.57      1.37           57.0
3배 이상 쏠린 클래스 | 05: ['COAD', 'THYM'] | 07: ['LUAD']


In [15]:
# 같은 모델이 train(OOF)에서는 쏠리지 않았는지 확인
#  → OOF에서는 정상인데 test에서만 쏠리면 test 쪽 차이(분포 차이) 때문이라는 근거
X_E = pd.concat([tr_any, tr_tr, tr_stats], axis=1)
r_E = run_cv(make_lr(C=0.1), X_E, y, verbose=False)   # 07 E 조건 그대로
oof_dist = pd.Series(le.inverse_transform(r_E['oof_proba'].argmax(1))).value_counts(normalize=True)
dist['oof_07'] = oof_dist
dist['oof_ratio_07'] = (dist['oof_07'] / dist['train']).round(2)
print(dist.loc[['SKCM', 'LUAD', 'LUSC', 'COAD', 'THYM'], ['train', 'oof_07', '07_E', 'oof_ratio_07', 'ratio_07']])

             train    oof_07      07_E  oof_ratio_07  ratio_07
SUBCLASS                                                      
SKCM      0.044509  0.037091  0.114690          0.83      2.58
LUAD      0.029673  0.024512  0.109191          0.83      3.68
LUSC      0.028705  0.034027  0.039277          1.19      1.37
COAD      0.035962  0.033543  0.073841          0.93      2.05
THYM      0.015804  0.056281  0.034171          3.56      2.16


### 결과

**1) 05 → 07 예측 분포 변화**
| 구분 | 클래스 (diff, train 변이 수 중앙값) |
| :--- | :--- |
| 많이 늘어난 클래스 | LUAD +0.095 (40.5), SKCM +0.089 (85.5), KIPAN +0.025 (15), LUSC +0.023 (57) |
| 많이 줄어든 클래스 | COAD −0.152 (23), THYM −0.032 (2), PCPG −0.011 (4), OV −0.011 (12) |
| 예외 | BLCA (54), STES (41), HNSC (31)는 변이가 많아도 거의 변화 없음 |

- Spearman(diff, train_med_mut) = **0.491** (중간 정도의 양의 상관)

**2) train 대비 예측 배율 (예측 비율 / train 비율)**
| 클래스 | 05 | 07 |
| :--- | :-: | :-: |
| LUAD | 0.46 | **3.68** |
| SKCM | 0.58 | **2.58** |
| LUSC | 0.57 | 1.37 |
| COAD | **6.27** | 2.05 |
| THYM | **4.20** | 2.16 |

- 3배 이상 쏠린 클래스: 05 → COAD, THYM / 07 → LUAD

**3) 같은 모델(07 E)의 train OOF 배율 vs test 배율**
| 클래스 | OOF | test |
| :--- | :-: | :-: |
| SKCM | 0.83 | 2.58 |
| LUAD | 0.83 | 3.68 |
| COAD | 0.93 | 2.05 |
| LUSC | 1.19 | 1.37 |
| THYM | **3.56** | 2.16 |

### 가설 판단
- **H5 채택**
  - 변이가 많은 LUAD, SKCM, LUSC가 05에서는 "덜 예측", 07에서는 "과하게 예측"으로 이동
  - diff와 클래스별 변이 수의 상관 0.491
  - 같은 모델이 train OOF에서는 SKCM, LUAD를 정상 비율(0.83배)로 예측하나 test에서만 2.6~3.7배
    → 쏠림이 test에서만 발생 = train/test 분포 차이에 stats가 반응한 것
- 단, BLCA, STES처럼 변이가 많아도 변하지 않은 클래스가 있어 완전한 비례 관계는 아님

### 추가로 알게 된 것
- **THYM 과예측은 별개 문제**: OOF에서도 3.56배 → test와 무관한 모델 자체의 쏠림
  → 변이 0개 행(train 94개, 15개 클래스 혼재, THYM 최다)을 모두 THYM으로 예측하는 것으로 추정
- **05도 COAD 6.27배로 쏠려 있었음**: stats가 없는데도 발생, 변이 수와 일관된 방향이 없어 원인 미확인
  (test를 더 분석해야 해서 누수 위험으로 중단)
- stats는 쏠림을 새로 만들었다기보다 쏠리는 대상을 COAD → LUAD, SKCM으로 옮긴 것에 가까움
- 전제: test의 실제 클래스 비율이 train과 비슷하다고 가정 (공개 정보 없음)

### 다음
- 확인 2 (H6): stats 제외(F) / 비율 피처(G)로 쏠림이 줄어드는지
- H7: 변이 0개 행 대응 → THYM 과예측 감소 (train OOF로 검증 가능, 제출 불필요)
- 제출 전 점검 지표 추가: 예측 배율 3배 이상인 클래스가 있으면 경고

-----

## 확인 2. 변이 개수 피처 제외 / 비율 피처로 교체

### 배경
- 확인 1(H5 채택): stats(변이 개수 피처)를 넣은 07 E 모델은 train OOF에서는 SKCM, LUAD를
  정상 비율(0.83배)로 예측하지만 test에서는 2.6~3.7배로 과예측
- → stats가 train/test 분포 차이(test 변이 비율 약 2배)에 반응한 것으로 판단

### 가설
- H6: 변이 "개수" 대신 "유무(행렬)"나 "비율"만 쓰면 test 예측 쏠림이 줄어든다
- H0: stats를 빼거나 비율로 바꿔도 test 예측 쏠림은 줄지 않는다

### 비교 조합
| 조합 | 구성 | 의미 |
| :--- | :--- | :--- |
| E (기준) | any + trunc + stats | 07 제출 모델 (CV 0.4188, LB 0.2706) |
| F | any + trunc | 개수 피처를 아예 뺌 |
| G | any + trunc + ratio | 개수 대신 비율만 사용 (침묵 비율, 절단 비율) |

- 비율은 변이가 전체적으로 2배가 되어도 크게 변하지 않음

### 검증 방법
- CV: 1차 필터로만 사용 (05의 0.3818보다 확실히 낮으면 탈락)
- **주 판단 지표: test 예측 배율** (예측 비율 / train 비율)
  - SKCM, LUAD 배율이 E(2.58, 3.68)보다 1에 가까워지는지
  - 3배 이상 쏠린 클래스 수
- 최종 확인: 쏠림이 덜한 쪽 1개만 제출해 LB 비교
- 비율 피처는 한 행 안에서만 계산 → Data Leakage 아님. 스케일러는 fold 안에서만 fit
- 예측 배율 비교는 모델 출력만 보는 것이며 test를 학습이나 튜닝에 쓰지 않음

In [16]:
# 비율 피처 만들기
# 행 단위 비율 피처: 변이 개수의 절대량에 덜 민감
def ratio_features(X_any, X_tr, stats):
    n_gene = np.maximum(X_any.sum(axis=1).values, 1)        # 변이 0개 행은 나눗셈 방지
    return pd.DataFrame({
        'silent_ratio': stats['silent_ratio'].values,       # 전체 변이 중 침묵 변이 비율
        'trunc_ratio':  X_tr.sum(axis=1).values / n_gene,   # 변이 유전자 중 절단 변이 유전자 비율
    }, index=X_any.index).astype(np.float32)

tr_ratio = ratio_features(tr_any, tr_tr, tr_stats)
te_ratio = ratio_features(te_any, te_tr, te_stats)
tr_ratio.describe().round(3)

,silent_ratio,trunc_ratio
count,6201.000,6201.000
mean,0.224,0.116
std,0.148,0.134
min,0.000,0.000
25%,0.137,0.000
50%,0.227,0.091
75%,0.308,0.161
max,1.000,1.000


In [17]:
# F, G CV 비교
# 표준화할 컬럼을 인자로 받도록 make_lr 재정의 (이전 셀의 전역 변수 STAT_COLS에 의존하지 않음)
def make_lr(C=0.1, scale_cols=None):
    def _make():
        lr = LogisticRegression(C=C, max_iter=3000, class_weight='balanced', random_state=SEED)
        if not scale_cols:
            return lr
        ct = ColumnTransformer([('sc', StandardScaler(), scale_cols)], remainder='passthrough')
        return make_pipeline(ct, lr)
    return _make

RATIO_COLS = list(tr_ratio.columns)
sets3 = {   # 이름: (train 피처, test 피처, 표준화할 컬럼)
    'F_any+trunc':       (pd.concat([tr_any, tr_tr], axis=1),           pd.concat([te_any, te_tr], axis=1),           None),
    'G_any+trunc+ratio': (pd.concat([tr_any, tr_tr, tr_ratio], axis=1), pd.concat([te_any, te_tr, te_ratio], axis=1), RATIO_COLS),
}

rows = []
for name, (Xs, _, cols) in sets3.items():
    for C in [0.05, 0.1]:
        r = run_cv(make_lr(C=C, scale_cols=cols), Xs, y, verbose=False)
        rows.append(dict(set=name, C=C, n_feat=Xs.shape[1], cv=r['cv_mean'], std=r['cv_std'], oof=r['oof_score']))
        print(rows[-1])
res7c = pd.DataFrame(rows).sort_values('cv', ascending=False)
res7c

{'set': 'F_any+trunc', 'C': 0.05, 'n_feat': 8768, 'cv': 0.38607237132684835, 'std': 0.00796433064249196, 'oof': 0.38773213118696315}
{'set': 'F_any+trunc', 'C': 0.1, 'n_feat': 8768, 'cv': 0.3921415273332035, 'std': 0.007361080242835493, 'oof': 0.39351389296880435}
{'set': 'G_any+trunc+ratio', 'C': 0.05, 'n_feat': 8770, 'cv': 0.3749447350530366, 'std': 0.013869948615416038, 'oof': 0.3762998910949483}
{'set': 'G_any+trunc+ratio', 'C': 0.1, 'n_feat': 8770, 'cv': 0.3858471345843154, 'std': 0.01229791752052593, 'oof': 0.3867438712275781}


,set,C,n_feat,cv,std,oof
1,F_any+trunc,0.10,8768,0.392142,0.007361,0.393514
0,F_any+trunc,0.05,8768,0.386072,0.007964,0.387732
3,G_any+trunc+ratio,0.10,8770,0.385847,0.012298,0.386744
2,G_any+trunc+ratio,0.05,8770,0.374945,0.013870,0.376300


### CV 결과

| 조합 | C=0.05 | C=0.1 | std (C=0.1) | 05 (0.3818) 대비 |
| :--- | :-: | :-: | :-: | :-: |
| E_any+trunc+stats | 0.4099 | 0.4188 | 0.0173 | +0.037 |
| **F_any+trunc** | 0.3861 | **0.3921** | **0.0074** | +0.010 |
| G_any+trunc+ratio | 0.3749 | 0.3858 | 0.0123 | +0.004 |

### 해석
- F, G 모두 1차 필터(05의 0.3818 이상)는 통과. 단 G는 겨우 넘는 수준
- **F의 std가 0.0074로 가장 작음** (E 0.0173의 절반 이하)
  → stats는 CV 점수를 올리는 동시에 fold 간 변동도 키우고 있었음
- **G가 F보다 낮음** (C=0.1 기준 −0.006)
  - 비율 피처 2개 추가가 오히려 손해
  - trunc_ratio는 25%가 0, 최댓값 1 → 변이가 1~2개뿐인 행에서 0 또는 1로 극단값이 나오는 잡음 피처일 가능성
- 두 조합 모두 최적 C가 범위 끝(0.1)이나, 이번 판단의 중심은 test 예측 배율이므로 C 추가 탐색은 보류

### 다음 단계 결정
- **F만 제출 파일 생성 후 예측 배율 확인**
  - F: CV가 더 높고 가장 안정적(std 최소), 구조도 가장 단순(개수·비율 피처 없음)
  - G: CV가 F보다 낮고, 추가한 비율 피처가 잡음일 가능성 → 이번에는 제외
- F의 test 배율이 E보다 1에 가까우면 오늘 마지막 1회로 제출

In [ ]:
# F 제출 파일 만들기
# F (any + trunc), C=0.1 → 파일만 생성. 다음 셀에서 예측 배율 확인 후 제출 여부 결정
X_F, X_F_test, _ = sets3['F_any+trunc']
r_F = run_cv(make_lr(C=0.1), X_F, y, X_F_test, verbose=False)   # 표준화할 컬럼 없음
path_F = save_submission(submission, r_F['test_proba'], le,
                         name="eunji_lr_any_trunc_C0.1", cv_score=r_F['cv_mean'])
oof_F = r_F['oof_proba'].argmax(1)
print(round(r_F['cv_mean'], 4), path_F.name)  

저장 완료: /Users/mac/web/dacon-cancer-subclass/submissions/1109_eunji_lr_any_trunc_C0.1_0.392.csv
0.3921 1109_eunji_lr_any_trunc_C0.1_0.392.csv


In [19]:
# 예측 배율 비교 (E vs F)
# test 배율 = 예측 비율 / train 비율. OOF 배율도 함께 (OOF는 정상인데 test만 크면 분포 차이 영향)
tr_rate = train.SUBCLASS.value_counts(normalize=True)
all_files = {'05': '1109_eunji_lr_l0_C0.1_0.382.csv',
             'E':  '1109_eunji_lr_any_trunc_stats_C0.1_0.419.csv',
             'F':  path_F.name}

ratio = pd.DataFrame({f'test_{k}': pd.read_csv(S / v).SUBCLASS.value_counts(normalize=True) / tr_rate
                      for k, v in all_files.items()}).fillna(0)
ratio['oof_F'] = pd.Series(le.inverse_transform(oof_F)).value_counts(normalize=True) / tr_rate
ratio['train_med_mut'] = dist['train_med_mut']

focus = ['SKCM', 'LUAD', 'LUSC', 'COAD', 'THYM']
print(ratio.loc[focus].round(2))
print('\n3배 이상 쏠린 클래스')
for c in ['test_05', 'test_E', 'test_F']:
    print(f'  {c}:', list(ratio.index[ratio[c] >= 3]))
print('\n배율이 1에서 벗어난 정도 (|log2 배율| 평균, 작을수록 train 분포와 비슷)')
print(np.log2(ratio[['test_05', 'test_E', 'test_F']].clip(lower=0.05)).abs().mean().round(3))

          test_05  test_E  test_F  oof_F  train_med_mut
SUBCLASS                                               
SKCM         0.58    2.58    0.66   0.78           85.5
LUAD         0.46    3.68    0.50   0.64           40.5
LUSC         0.57    1.37    0.63   0.87           57.0
COAD         6.27    2.05    4.71   0.92           23.0
THYM         4.20    2.16    4.60   6.67            2.0

3배 이상 쏠린 클래스
  test_05: ['COAD', 'THYM']
  test_E: ['LUAD']
  test_F: ['COAD', 'THYM']

배율이 1에서 벗어난 정도 (|log2 배율| 평균, 작을수록 train 분포와 비슷)
test_05    0.820
test_E     0.897
test_F     0.781
dtype: float64


### 결과

**1) CV**
| 조합 | 최적 C | CV (± std) |
| :--- | :-: | :-: |
| E_any+trunc+stats | 0.1 | 0.4188 ± 0.0173 |
| F_any+trunc | 0.1 | 0.3921 ± 0.0074 |
| G_any+trunc+ratio | 0.1 | 0.3858 ± 0.0123 (CV가 낮아 제출 파일을 만들지 않음) |

**2) test 예측 배율 (예측 비율 / train 비율)**
| 클래스 | 05 | E | F | F의 OOF |
| :--- | :-: | :-: | :-: | :-: |
| SKCM | 0.58 | 2.58 | 0.66 | 0.78 |
| LUAD | 0.46 | 3.68 | 0.50 | 0.64 |
| LUSC | 0.57 | 1.37 | 0.63 | 0.87 |
| COAD | 6.27 | 2.05 | 4.71 | 0.92 |
| THYM | 4.20 | 2.16 | 4.60 | 6.67 |
| 3배 이상 클래스 | COAD, THYM | LUAD | COAD, THYM | — |
| \|log2 배율\| 평균 | 0.820 | 0.897 | **0.781** | — |

**3) 제출**
- 선택: F (test 배율이 가장 덜 쏠림, CV std 최소)
- LB: **0.2700** (05 0.2739 대비 −0.004, E 0.2706 대비 −0.001)

### 가설 판단
- **H6 부분 채택**
  - stats를 빼면 SKCM, LUAD 과예측은 사라짐 (2.58 → 0.66, 3.68 → 0.50)
  - 그러나 COAD, THYM 쏠림이 05 수준으로 돌아옴 (4.71, 4.60)
  - 전체 쏠림 정도는 F가 가장 작지만(0.781) 05(0.820)와 큰 차이 없음
  - → stats 제거는 쏠림을 "없앤" 것이 아니라 다시 "옮긴" 것에 가까움

### 추가로 알게 된 것
- **COAD 쏠림은 test에서만 발생** (OOF 0.92 → test 4.71)
  → stats와 무관하게 이진 행렬 LR이 test에서 COAD로 쏠림. 원인 미확인 (test 추가 분석은 누수 위험으로 중단)
- **THYM 쏠림은 모델 자체 문제** (OOF에서도 6.67배)
  → 변이 0개 행을 THYM으로 몰아 예측하는 것으로 추정 → H7에서 train OOF로 검증
- 05와 F의 차이는 trunc 행렬뿐 → LB −0.004로 **trunc 행렬도 test에서는 효과 없음**

### 다음
- 운영진 문의: test 데이터 생성 방식(빈칸, 변이 비율 차이)이 train과 다른지
- H7: 변이 0개 행 대응 → THYM 과예측 감소 (train OOF로 검증, 제출 불필요)
- H8: 점수를 단순 합산하지 않는 모델(LightGBM 등)은 test 쏠림(COAD)이 덜하다
  → 04 LightGBM 제출 파일을 만들어 배율 비교. LR 계열이 LB 0.27에 막혀 있으므로 제출 1회 후보
- 제출 전 점검 지표(예측 배율 3배 이상 클래스, |log2 배율| 평균)는 참고용으로만 사용 (LB 예측력 없음, 아래 종합 결론 참고)

-----

## 07 종합 결론 (LB 기준)

| 제출 | 구성 | CV | LB | 05 대비 LB |
| :--- | :--- | :-: | :-: | :-: |
| 05 | any | 0.3818 | **0.2739** | — |
| E | any + trunc + stats | 0.4188 | 0.2706 | −0.003 |
| F | any + trunc | 0.3921 | 0.2700 | −0.004 |

- 05 / E / F의 LB가 0.270~0.274로 모두 비슷 → **stats도 trunc도 test에서는 효과 없음**
  - 0.003~0.004 차이는 오차 범위로 판단. "떨어뜨렸다"보다 "움직이지 못했다"로 해석
- E와 F는 예측 분포가 크게 다른데(E: LUAD·SKCM 쏠림, F: COAD·THYM 쏠림) LB는 같음
  → 틀리는 클래스만 바뀌고 틀리는 양은 그대로
- 쏠림 지표(|log2 배율|)는 F가 최소(0.781)였지만 LB도 최저 → **이 지표로는 LB 예측 불가**
  - test 클래스 비율이 train과 같다는 가정에 기대는 지표이므로 제출 기준이 아닌 참고용
- CV 개선폭: E +0.037, F +0.010 → LB 개선폭: 둘 다 0 → **이 대회에서 CV만으로 피처를 고르면 위험** (재확인)

### 현재 판단
- LR + 이진 피처 계열은 LB 0.27 근처가 한계로 보임
- 피처 추가보다 우선할 것
  1. test 분포 차이의 원인 확인 (운영진 문의)
  2. 모델 계열 변경 (H8: LightGBM 등)
  3. 변이 0개 행 대응 (H7, train OOF로 검증)
- 최고 LB는 05 (0.2739) 유지